In [ ]:
%config InlineBackend.figure_formats = ['svg']
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['FreeSans']
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import re, sys, torch
from types import SimpleNamespace
from torch.utils.data import DataLoader
from tqdm import tqdm
from joblib import Parallel, delayed

sys.path.insert(0, '../4_BCQf')
from model import BCQf, all_subactions_vec
from data import EpisodicBuffer as EpisodicBufferFF, remap_rewards
from evaluate import EpisodicBufferF, offline_evaluation_F

state_dim = 64   
num_actions = 25
horizon = 20

## 1. Non-Shifted (Original Tang Alignment) — Top 20 ESS

In [ ]:
# Load non-shifted test data
test_orig = EpisodicBufferF(state_dim, num_actions, horizon)
test_orig.load('../data/episodes+encoded_state+knn_pibs_factored/test_data.pt')
test_orig.reward = remap_rewards(
    test_orig.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)
test_orig_loader = DataLoader(test_orig, batch_size=len(test_orig), shuffle=False)
test_batch_orig = next(iter(test_orig_loader))

In [ ]:
# Find top 10 ESS checkpoints per version (non-shifted)
logdir_orig = "../4_BCQf/logs/mimic_dBCQf"

orig_top10_list = []
for ver in range(40):
    try:
        text = open(f"{logdir_orig}/version_{ver}/hparams.yaml").read()
        thresh = float(re.search(r"threshold: ([\\d.]+)", text).group(1))
        seed = int(re.search(r"seed: (\\d+)", text).group(1))
        df = pd.read_csv(f"{logdir_orig}/version_{ver}/metrics.csv")
        valid = df.dropna(subset=["val_wis", "val_ess"])
        if len(valid) > 0:
            top5 = valid.nlargest(10, "val_ess")
            for _, row in top5.iterrows():
                orig_top10_list.append({
                    "version": ver, "threshold": thresh, "seed": seed,
                    "val_wis": row["val_wis"], "val_ess": row["val_ess"],
                    "iteration": row["iteration"],
                })
    except Exception as e:
        print(f"Version {ver} skipped: {e}")

top10_orig = pd.DataFrame(orig_top10_list)
print(f"Non-shifted: {len(top10_orig)} checkpoints from {top10_orig["version"].nunique()} versions")
print(top10_orig.groupby("version").size().describe())  # should be 5 per version


In [ ]:
# Evaluate all top-10 checkpoints on test set (GPU)
orig_test_results = []
for _, row in tqdm(top10_orig.iterrows(), total=len(top10_orig), desc="Orig test"):
    ver = int(row["version"])
    best_iter = int(row["iteration"])
    ckpt_step = (best_iter // 100) * 100
    ckpt_path = f"{logdir_orig}/version_{ver}/step={ckpt_step}.ckpt"
    model = BCQf.load_from_checkpoint(ckpt_path, map_location=None, weights_only=False)
    model.eval()
    model.all_subactions_vec = all_subactions_vec
    w, e = offline_evaluation_F(model, test_batch_orig.to(model.device), weighted=True, eps=0.01)
    orig_test_results.append({
        "model": "BCQf (orig)",
        "version": ver, "threshold": row["threshold"], "seed": row["seed"],
        "val_wis": row["val_wis"], "val_ess": row["val_ess"], "best_iter": best_iter,
        "test_wis": w, "test_ess": e,
    })

df_orig = pd.DataFrame(orig_test_results)
print(f"\nNon-shifted: {len(df_orig)} models tested")
print(df_orig.nlargest(10, "test_wis")[["version","threshold","seed","best_iter","val_ess","test_wis","test_ess"]].to_string(index=False))


## 2. Shifted Alignment — Top 20 ESS

In [ ]:
# Load shifted test data (need state_dim=64 for shifted)
state_dim_s = 64
test_shifted = EpisodicBufferF(state_dim_s, num_actions, horizon)
test_shifted.load('../data/episodes+encoded_state+knn_pibs_factored/shifted_test_data.pt')
test_shifted.reward = remap_rewards(
    test_shifted.reward,
    SimpleNamespace(**{'R_immed': 0.0, 'R_death': 0.0, 'R_disch': 100.0})
)
test_s_loader = DataLoader(test_shifted, batch_size=len(test_shifted), shuffle=False)
test_batch_s = next(iter(test_s_loader))

In [ ]:
# Find top 10 ESS checkpoints per version (shifted)
logdir_shifted = "../4_BCQf/logs_shifted/mimic_dBCQf_shifted"

shifted_top10_list = []
for ver in range(40):
    try:
        text = open(f"{logdir_shifted}/version_{ver}/hparams.yaml").read()
        thresh = float(re.search(r"threshold: ([\\d.]+)", text).group(1))
        seed = int(re.search(r"seed: (\\d+)", text).group(1))
        df = pd.read_csv(f"{logdir_shifted}/version_{ver}/metrics.csv")
        valid = df.dropna(subset=["val_wis", "val_ess"])
        if len(valid) > 0:
            top5 = valid.nlargest(10, "val_ess")
            for _, row in top5.iterrows():
                shifted_top10_list.append({
                    "version": ver, "threshold": thresh, "seed": seed,
                    "val_wis": row["val_wis"], "val_ess": row["val_ess"],
                    "iteration": row["iteration"],
                })
    except Exception as e:
        print(f"Version {ver} skipped: {e}")

top10_shifted = pd.DataFrame(shifted_top10_list)
print(f"Shifted: {len(top10_shifted)} checkpoints from {top10_shifted["version"].nunique()} versions")


In [ ]:
# Evaluate all top-10 checkpoints on shifted test set (GPU)
shifted_test_results = []
for _, row in tqdm(top10_shifted.iterrows(), total=len(top10_shifted), desc="Shifted test"):
    ver = int(row["version"])
    best_iter = int(row["iteration"])
    ckpt_step = (best_iter // 100) * 100
    ckpt_path = f"{logdir_shifted}/version_{ver}/step={ckpt_step}.ckpt"
    model = BCQf.load_from_checkpoint(ckpt_path, map_location=None, weights_only=False)
    model.eval()
    model.all_subactions_vec = all_subactions_vec
    w, e = offline_evaluation_F(model, test_batch_s.to(model.device), weighted=True, eps=0.01)
    shifted_test_results.append({
        "model": "BCQf (shifted)",
        "version": ver, "threshold": row["threshold"], "seed": row["seed"],
        "val_wis": row["val_wis"], "val_ess": row["val_ess"], "best_iter": best_iter,
        "test_wis": w, "test_ess": e,
    })

df_shifted = pd.DataFrame(shifted_test_results)
print(f"\nShifted: {len(df_shifted)} models tested")
print(df_shifted.nlargest(10, "test_wis")[["version","threshold","seed","best_iter","val_ess","test_wis","test_ess"]].to_string(index=False))


## 3. Bootstrap CI — Clinician vs Best BCQf (Orig) vs Best BCQf (Shifted)

In [ ]:
# Pick best models (by test WIS from all 200)
best_orig = df_orig.loc[df_orig["test_wis"].idxmax()]
best_shifted = df_shifted.loc[df_shifted["test_wis"].idxmax()]

print(f"Best original: thresh={best_orig["threshold"]:.4f} seed={int(best_orig["seed"])} v{int(best_orig["version"])} iter={int(best_orig["best_iter"])} test_wis={best_orig["test_wis"]:.2f}")
print(f"Best shifted:  thresh={best_shifted["threshold"]:.4f} seed={int(best_shifted["seed"])} v{int(best_shifted["version"])} iter={int(best_shifted["best_iter"])} test_wis={best_shifted["test_wis"]:.2f}")

# Load best models (GPU)
orig_ckpt = f"{logdir_orig}/version_{int(best_orig["version"])}/step={(int(best_orig["best_iter"])//100)*100}.ckpt"
s_ckpt    = f"{logdir_shifted}/version_{int(best_shifted["version"])}/step={(int(best_shifted["best_iter"])//100)*100}.ckpt"

model_orig = BCQf.load_from_checkpoint(orig_ckpt, map_location=None, weights_only=False)
model_orig.eval()
model_orig.all_subactions_vec = all_subactions_vec

model_shifted = BCQf.load_from_checkpoint(s_ckpt, map_location=None, weights_only=False)
model_shifted.eval()
model_shifted.all_subactions_vec = all_subactions_vec


In [ ]:
# Bootstrap: clinician reward (non-shifted test)
def bootstrap_clinician_orig(i):
    n = len(test_orig)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = test_orig[idx]
    return batch[4].sum(axis=1).float().mean()

def bootstrap_clinician_shifted(i):
    n = len(test_shifted)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    batch = test_shifted[idx]
    return batch[4].sum(axis=1).float().mean()

# Bootstrap: BCQf WIS
def bootstrap_wis_orig(i):
    n = len(test_orig)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    return offline_evaluation_F(model_orig, test_orig[idx], weighted=True, eps=0.01)

def bootstrap_wis_shifted(i):
    n = len(test_shifted)
    idx = np.random.default_rng(seed=i).choice(n, n, replace=True)
    return offline_evaluation_F(model_shifted, test_shifted[idx], weighted=True, eps=0.01)

In [ ]:
print('Running 100 bootstraps...')

boot_clinician_orig = Parallel(n_jobs=6)(delayed(bootstrap_clinician_orig)(i) for i in tqdm(range(100), desc='Clinician orig'))
boot_clinician_shifted = Parallel(n_jobs=6)(delayed(bootstrap_clinician_shifted)(i) for i in tqdm(range(100), desc='Clinician shifted'))
boot_wis_orig = zip(*Parallel(n_jobs=6)(delayed(bootstrap_wis_orig)(i) for i in tqdm(range(100), desc='BCQf orig WIS')))
boot_wis_s = zip(*Parallel(n_jobs=6)(delayed(bootstrap_wis_shifted)(i) for i in tqdm(range(100), desc='BCQf shifted WIS')))
boot_orig_wis, boot_orig_ess = map(list, boot_wis_orig)
boot_s_wis, boot_s_ess = map(list, boot_wis_s)

## 4. Final Comparison

In [ ]:
# Single-step WIS/ESS (non-bootstrap)
wis_orig, ess_orig = offline_evaluation_F(model_orig, test_batch_orig, weighted=True, eps=0.01)
wis_shifted, ess_shifted = offline_evaluation_F(model_shifted, test_batch_s, weighted=True, eps=0.01)

clinician_orig_wis = test_orig.reward.sum(axis=1).mean()
clinician_shifted_wis = test_shifted.reward.sum(axis=1).mean()

print(f"{"="*55}")
print(f'{"Model":<25} {"Test WIS":>15} {"Test ESS":>15}')
print(f"{"="*55}")
print(f'{"Clinician (non-shifted)":<25} {clinician_orig_wis:15.2f} {"—":>15}')
print(f'{"Clinician (shifted)":<25} {clinician_shifted_wis:15.2f} {"—":>15}')
print(f'{"BCQf (non-shifted)":<25} {wis_orig:15.2f} {ess_orig:15.1f}')
print(f'{"BCQf (shifted)":<25} {wis_shifted:15.2f} {ess_shifted:15.1f}')
print(f"{"="*55}")
print()
print(f"Best original:  thresh={best_orig.iloc[0]["threshold"]:.4f} seed={int(best_orig.iloc[0]["seed"])} iter={best_orig.iloc[0]["best_iter"]}")
print(f"Best shifted:   thresh={best_shifted.iloc[0]["threshold"]:.4f} seed={int(best_shifted.iloc[0]["seed"])} iter={best_shifted.iloc[0]["best_iter"]}")
print()
print("Bootstrap 95% CI (100 samples):")
print(f"  Clinician (non-shifted): {clinician_orig_wis:.2f} ± {np.std(boot_clinician_orig):.2f}")
print(f"  Clinician (shifted):     {clinician_shifted_wis:.2f} ± {np.std(boot_clinician_shifted):.2f}")
print(f"  BCQf (non-shifted):      WIS={wis_orig:.2f} ± {np.std(boot_orig_wis):.2f}  |  ESS={ess_orig:.1f} ± {np.std(boot_orig_ess):.1f}")
print(f"  BCQf (shifted):          WIS={wis_shifted:.2f} ± {np.std(boot_s_wis):.2f}  |  ESS={ess_shifted:.1f} ± {np.std(boot_s_ess):.1f}")


In [ ]:
# Save combined results
pd.concat([df_orig, df_shifted]).to_csv('combined_top20_results.csv', index=False)
print('Saved combined_top20_results.csv')